In [3]:
import pandas as pd


file_path = "/Users/nidhichaubey/Desktop/TMDB_all_movies.csv"  # update this

df = pd.read_csv(file_path)

df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9730.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.300,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, English","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8613.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.900,2819.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,English,"Sammi Davis, Marc Lawrence, Alicia Witt, Madon...","Robert Rodriguez, Allison Anders, Quentin Tara...","Rodrigo García, Guillermo Navarro, Phil Parmet...","Allison Anders, Robert Rodriguez, Alexandre Ro...","Quentin Tarantino, Alexandre Rockwell, Lawrenc...",Combustible Edison,6.7,116820.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.500,370.0,Released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,...,English,"Jeremy Piven, Lydell M. Cheshier, Michael DeLo...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Lloyd Segan, Gene Levy, Marilyn Vance",Alan Silvestri,6.6,21019.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.200,30.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Timo Novotny, Michael Glawogger","Timo Novotny, Ulrich Gehmacher",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


In [4]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (1184452, 28)

Columns:
Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'budget', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'cast', 'director', 'director_of_photography', 'writers', 'producers',
       'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path'],
      dtype='object')

Data Types:
id                           int64
title                       object
vote_average               float64
vote_count                 float64
status                      object
release_date                object
revenue                    float64
runtime                    float64
budget                     float64
imdb_id                     object
original_language           object
original_title              object
overview                    object
popularity                 float64


In [5]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
print("\nCleaned Column Names:")
print(df.columns)



Cleaned Column Names:
Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'budget', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'cast', 'director', 'director_of_photography', 'writers', 'producers',
       'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path'],
      dtype='object')


In [6]:
# Remove completely empty rows
df = df.dropna(how="all")

# Remove duplicates
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)

Shape after cleaning: (1184452, 28)


In [7]:
# Numeric columns → fill with median
for col in df.select_dtypes(include="number"):
    df[col] = df[col].fillna(df[col].median())

# Text columns → fill with "unknown"
for col in df.select_dtypes(include="object"):
    df[col] = df[col].fillna("unknown")

print("Shape after filling missing values:", df.shape)

Shape after filling missing values: (1184452, 28)


In [8]:
# Example fixes (adjust based on your columns)

if "release_date" in df.columns:
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

if "rating" in df.columns:
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

print("Shape after data type conversion:", df.shape)

Shape after data type conversion: (1184452, 28)


In [9]:
for col in df.select_dtypes(include="object"):
    df[col] = df[col].str.strip().str.lower()

print("Shape after string cleaning:", df.shape)

Shape after string cleaning: (1184452, 28)


In [14]:
if "rating" in df.columns:
    df = df[(df["rating"] >= 0) & (df["rating"] <= 10)]

print("Shape after filtering rating values:", df.shape)

Shape after filtering rating values: (1184452, 28)


In [15]:
df.to_parquet("clean_movies.parquet", index=False)

print("Saved successfully")

Saved successfully


In [19]:
import duckdb

con = duckdb.connect()

df_sample = con.execute("""
    SELECT * 
    FROM 'clean_movies.parquet'
    LIMIT 5
""").fetchdf()

df_sample

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,ariel,7.106,371.0,released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"kari helaseppä, jaakko talaskivi, mikko remes,...",aki kaurismäki,timo salminen,aki kaurismäki,aki kaurismäki,unknown,7.4,9730.0,/ojdg0pgvs6r9xyfodrct2kdi6wc.jpg
1,3,shadows in paradise,7.300,435.0,released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, english","ari korhonen, mari rantasila, erkki rissanen, ...",aki kaurismäki,timo salminen,aki kaurismäki,mika kaurismäki,unknown,7.4,8613.0,/nj01hspawpof0mjmlgfjulyjurn.jpg
2,5,four rooms,5.900,2819.0,released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,english,"sammi davis, marc lawrence, alicia witt, madon...","robert rodriguez, allison anders, quentin tara...","rodrigo garcía, guillermo navarro, phil parmet...","allison anders, robert rodriguez, alexandre ro...","quentin tarantino, alexandre rockwell, lawrenc...",combustible edison,6.7,116820.0,/75ahn1noyxh4m7l5shoeq6ngykp.jpg
3,6,judgment night,6.500,370.0,released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,...,english,"jeremy piven, lydell m. cheshier, michael delo...",stephen hopkins,peter levy,"lewis colick, jere cunningham","lloyd segan, gene levy, marilyn vance",alan silvestri,6.6,21019.0,/3rvvps9ypm5hb2f4hyinijvtdam.jpg
4,8,life in loops (a megacities rmx),7.200,30.0,released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,"english, हिन्दी, 日本語, pусский, español",unknown,timo novotny,wolfgang thaler,"timo novotny, michael glawogger","timo novotny, ulrich gehmacher",unknown,8.1,285.0,/7ln81brnpr2wqxuitzxecice1lc.jpg


In [28]:
import duckdb
con = duckdb.connect()
con.execute("CREATE OR REPLACE TABLE clean_movies AS SELECT * FROM clean_movies.parquet")
con.execute("DESCRIBE clean_movies").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,vote_average,DOUBLE,YES,None,None,None
3,vote_count,DOUBLE,YES,None,None,None
4,status,VARCHAR,YES,None,None,None
5,release_date,TIMESTAMP_NS,YES,None,None,None
6,revenue,DOUBLE,YES,None,None,None
7,runtime,DOUBLE,YES,None,None,None
8,budget,DOUBLE,YES,None,None,None
9,imdb_id,VARCHAR,YES,None,None,None


In [29]:
con.execute("SELECT * FROM clean_movies LIMIT 2").df()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,ariel,7.106,371.0,released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"kari helaseppä, jaakko talaskivi, mikko remes,...",aki kaurismäki,timo salminen,aki kaurismäki,aki kaurismäki,unknown,7.4,9730.0,/ojdg0pgvs6r9xyfodrct2kdi6wc.jpg
1,3,shadows in paradise,7.300,435.0,released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, english","ari korhonen, mari rantasila, erkki rissanen, ...",aki kaurismäki,timo salminen,aki kaurismäki,mika kaurismäki,unknown,7.4,8613.0,/nj01hspawpof0mjmlgfjulyjurn.jpg


In [36]:
con.execute("""
    COPY (
        SELECT DISTINCT
            CAST(id AS VARCHAR)          AS movie_id,
            title,
            CAST(release_date AS VARCHAR) AS release_date,
            vote_average,
            vote_count,
            runtime,
            revenue,
            budget,
            original_language,
            overview,
            tagline,
            popularity,
            imdb_rating
        FROM clean_movies
        WHERE id IS NOT NULL AND title IS NOT NULL
    ) TO '/Users/nidhichaubey/Masters_Project/big_data_capstone_project/data/Movie_nodes.csv' (HEADER, FORMAT CSV)
""")
print("✅ Movie nodes exported:", con.execute("SELECT COUNT(*) FROM clean_movies").fetchone()[0])

✅ Movie nodes exported: 1184452


In [42]:
con.execute("""
    CREATE OR REPLACE TABLE actors_clean AS
    SELECT DISTINCT
        trim(actor_name) AS actor_id,
        trim(actor_name) AS name
    FROM (
        SELECT unnest(string_split("cast", ',')) AS actor_name
        FROM clean_movies
        WHERE "cast" IS NOT NULL AND "cast" != ''
    )
    WHERE trim(actor_name) != ''
""")

con.execute("""
    COPY actors_clean TO '/Users/nidhichaubey/Masters_Project/big_data_capstone_project/data/Actor_nodes.csv' (HEADER, FORMAT CSV)
""")
print("✅ Actor nodes exported:", con.execute("SELECT COUNT(*) FROM actors_clean").fetchone()[0])

✅ Actor nodes exported: 2100314


In [43]:
con.execute("""
    CREATE OR REPLACE TABLE directed_by AS
    SELECT DISTINCT
        CAST(id AS VARCHAR) AS movie_id,
        trim(director)      AS director_id
    FROM clean_movies
    WHERE director IS NOT NULL AND trim(director) != ''
""")

con.execute("""
    COPY directed_by TO '/Users/nidhichaubey/Masters_Project/big_data_capstone_project/data/DIRECTED_BY.csv' (HEADER, FORMAT CSV)
""")
print("✅ DIRECTED_BY relationships exported:", con.execute("SELECT COUNT(*) FROM directed_by").fetchone()[0])

✅ DIRECTED_BY relationships exported: 1184452


In [44]:
con.execute("""
    CREATE OR REPLACE TABLE acted_in AS
    SELECT DISTINCT
        CAST(m.id AS VARCHAR) AS movie_id,
        trim(actor_name)      AS actor_id
    FROM clean_movies m,
         unnest(string_split(m.cast, ',')) AS t(actor_name)
    WHERE m.cast IS NOT NULL
      AND trim(actor_name) != ''
""")

con.execute("""
    COPY acted_in TO '/Users/nidhichaubey/Masters_Project/big_data_capstone_project/data/ACTED_IN.csv' (HEADER, FORMAT CSV)
""")
print("✅ ACTED_IN relationships exported:", con.execute("SELECT COUNT(*) FROM acted_in").fetchone()[0])

✅ ACTED_IN relationships exported: 7762530


In [45]:
con.execute("""
    CREATE OR REPLACE TABLE directors_clean AS
    SELECT DISTINCT
        trim(director) AS director_id,
        trim(director) AS name
    FROM clean_movies
    WHERE director IS NOT NULL AND trim(director) != ''
""")

con.execute("""
    COPY directors_clean TO '/Users/nidhichaubey/Masters_Project/big_data_capstone_project/data/Director_nodes.csv' (HEADER, FORMAT CSV)
""")
print("✅ Director nodes exported:", con.execute("SELECT COUNT(*) FROM directors_clean").fetchone()[0])

✅ Director nodes exported: 411256
